# KBStats Matchday-Qualified Last-Five Average Points

This notebook uses the same matchday-dependent appearance qualification as the SofaScore player-average-ratings notebook. It calculates the last-five average from played history slots only: unplayed slots are retained as `null` in `last5Points` but do not contribute points or to the denominator.

Qualified players without a played slot in the five-slot window are retained with `last5AveragePoints: null`. `history[0]` is the latest slot.

In [1]:
from __future__ import annotations

import json
import math
import re
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display


def locate_project_root() -> Path:
    starts = []
    notebook_path = globals().get('__vsc_ipynb_file__')
    if isinstance(notebook_path, str) and notebook_path.strip():
        starts.append(Path(notebook_path).expanduser().resolve().parent)
    starts.append(Path.cwd().resolve())
    for start in starts:
        for candidate in (start, *start.parents):
            if (candidate / 'project_paths.py').is_file():
                return candidate
    raise FileNotFoundError('Could not locate project_paths.py. Start Jupyter from the project root.')


PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from project_paths import (
    DERIVED_KBSTATS_MATCHDAY_QUALIFIED_LAST_5_AVERAGE_PLAYERS_DIR,
    KBSTATS_PLAYERS_DIR,
    ensure_directory,
)

EARLY_SEASON_MAX_MATCHDAY = 3
EARLY_SEASON_MINIMUM_PLAYED_MATCHES = 2
MINIMUM_PLAYED_MATCHES = 3
MINIMUM_BUNDESLIGA_MATCHDAY = 1
MAXIMUM_BUNDESLIGA_MATCHDAY = 34
MATCH_SLOT_COUNT = 5
KBSTATS_FILENAME_RE = re.compile(
    r'^kbstats_players_(?P<date>\d{8})_(?P<time>\d{6})_(?P<offset>[+-]\d{4})\.json$'
)


def prompt_bundesliga_matchday() -> int:
    raw_matchday = input(f'Current Bundesliga matchday ({MINIMUM_BUNDESLIGA_MATCHDAY}-{MAXIMUM_BUNDESLIGA_MATCHDAY}): ').strip()
    try:
        matchday = int(raw_matchday)
    except ValueError as exc:
        raise ValueError('Bundesliga matchday must be a whole number.') from exc
    if not MINIMUM_BUNDESLIGA_MATCHDAY <= matchday <= MAXIMUM_BUNDESLIGA_MATCHDAY:
        raise ValueError(f'Bundesliga matchday must be between {MINIMUM_BUNDESLIGA_MATCHDAY} and {MAXIMUM_BUNDESLIGA_MATCHDAY}.')
    return matchday


BUNDESLIGA_MATCHDAY = prompt_bundesliga_matchday()


Current Bundesliga matchday (1-34):  1


In [2]:
def parse_snapshot_timestamp(path: Path) -> datetime:
    match = KBSTATS_FILENAME_RE.fullmatch(path.name)
    if match is None:
        raise ValueError(f'Unsupported KBStats filename: {path.name}')
    parsed = datetime.strptime(f"{match.group('date')}_{match.group('time')}_{match.group('offset')}", '%Y%m%d_%H%M%S_%z')
    return parsed.astimezone(timezone.utc)


def select_latest_snapshot(directory: Path) -> Path:
    candidates = []
    for path in sorted(directory.glob('kbstats_players_*.json')):
        try:
            candidates.append((parse_snapshot_timestamp(path), path))
        except ValueError as exc:
            warnings.warn(f'Ignoring {path.name}: {exc}', stacklevel=2)
    if not candidates:
        raise FileNotFoundError(f'No valid kbstats_players_*.json files in {directory}')
    latest = max(timestamp for timestamp, _ in candidates)
    latest_paths = [path for timestamp, path in candidates if timestamp == latest]
    if len(latest_paths) != 1:
        raise RuntimeError('Multiple KBStats files encode the same latest instant.')
    return latest_paths[0]


def is_finite_number(value: Any) -> bool:
    return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(float(value))


def played_in_history_slot(player: dict[str, Any], slot_index: int) -> bool:
    history = player.get('history')
    return bool(isinstance(history, list) and len(history) > slot_index and isinstance(history[slot_index], dict) and history[slot_index].get('hasPlayed') is True)


def qualification_rule_description(matchday: int) -> str:
    if matchday <= EARLY_SEASON_MAX_MATCHDAY:
        return 'at least 2 played matches or played in the latest history slot'
    return 'at least 3 played matches or played in both latest history slots'


def player_qualifies(player: dict[str, Any], matchday: int) -> bool:
    games_played = player.get('gamesPlayed')
    if not is_finite_number(games_played):
        return False
    if matchday <= EARLY_SEASON_MAX_MATCHDAY:
        return games_played >= EARLY_SEASON_MINIMUM_PLAYED_MATCHES or played_in_history_slot(player, 0)
    return games_played >= MINIMUM_PLAYED_MATCHES or (played_in_history_slot(player, 0) and played_in_history_slot(player, 1))


def derive_last_five(player: dict[str, Any]) -> dict[str, Any]:
    history = player.get('history')
    if not isinstance(history, list) or len(history) < MATCH_SLOT_COUNT:
        raise ValueError('history must contain five match slots')
    points, played_slots = [], 0
    for slot_index, slot in enumerate(history[:MATCH_SLOT_COUNT]):
        if not isinstance(slot, dict):
            raise ValueError(f'history slot {slot_index} is not an object')
        if slot.get('hasPlayed') is True:
            value = slot.get('points')
            if not is_finite_number(value):
                raise ValueError(f'played history slot {slot_index} has non-numeric points')
            points.append(float(value))
            played_slots += 1
        else:
            points.append(None)
    return {'last5Points': points, 'last5PlayedSlots': played_slots, 'last5AveragePoints': sum(point for point in points if point is not None) / played_slots if played_slots else None}


selected_input_path = select_latest_snapshot(KBSTATS_PLAYERS_DIR)
try:
    raw_players = json.loads(selected_input_path.read_text(encoding='utf-8'))
except (OSError, UnicodeDecodeError, json.JSONDecodeError) as exc:
    raise ValueError(f'Could not load KBStats input {selected_input_path}: {exc}') from exc
if not isinstance(raw_players, list):
    raise TypeError(f'Expected a JSON list of players, got {type(raw_players).__name__}.')

valid_players, invalid_records = [], []
for source_index, player in enumerate(raw_players):
    if not isinstance(player, dict):
        invalid_records.append({'source_index': source_index, 'reason': 'record is not an object'})
    elif not is_finite_number(player.get('gamesPlayed')):
        invalid_records.append({'source_index': source_index, 'player_id': player.get('id'), 'reason': 'gamesPlayed is not finite numeric data'})
    else:
        valid_players.append(player)

qualified_players, invalid_last_five_records = [], []
for player in valid_players:
    if not player_qualifies(player, BUNDESLIGA_MATCHDAY):
        continue
    try:
        qualified_players.append({**player, **derive_last_five(player)})
    except ValueError as exc:
        invalid_last_five_records.append({'player_id': player.get('id'), 'reason': str(exc)})

qualified_players.sort(key=lambda player: (player['last5AveragePoints'] is None, -(player['last5AveragePoints'] or 0), -float(player.get('averagePoints') or 0), str(player.get('name') or '').casefold()))
display_df = pd.DataFrame([
    {'player_id': player.get('id'), 'name': player.get('name'), 'team_id': player.get('teamId'), 'position': player.get('position'), 'last_5_points': player['last5Points'], 'played_slots': player['last5PlayedSlots'], 'last_5_average_points': player['last5AveragePoints'], 'season_average_points': player.get('averagePoints'), 'games_played': player.get('gamesPlayed')}
    for player in qualified_players
])
print(f'Selected input: {selected_input_path.name}')
print(f'Matchday {BUNDESLIGA_MATCHDAY} rule: {qualification_rule_description(BUNDESLIGA_MATCHDAY)}')
print(f'Source records: {len(raw_players):,}; valid records: {len(valid_players):,}; invalid base records: {len(invalid_records):,}; invalid last-five records: {len(invalid_last_five_records):,}; qualified players: {len(qualified_players):,}')
display(display_df)


Selected input: kbstats_players_20260818_204143_+0200.json
Matchday 1 rule: at least 2 played matches or played in the latest history slot
Source records: 468; valid records: 310; invalid base records: 158; invalid last-five records: 0; qualified players: 304


,player_id,name,team_id,position,last_5_points,played_slots,last_5_average_points,season_average_points,games_played
0,3120,Tom Bischof,2,3,"[268.0, 228.0, None, None, None]",2,248.00,118.0,26.0
1,2736,Ramy Bensebaini,3,2,"[None, None, None, 364.0, 106.0]",2,235.00,128.0,21.0
2,12368,Sander Tangvik,6,1,"[220.0, None, None, None, None]",1,220.00,220.0,1.0
3,1685,Joshua Kimmich,2,3,"[317.0, 238.0, 83.0, None, 215.0]",4,213.25,186.0,29.0
4,8329,Michael Olise,2,3,"[81.0, 334.0, 290.0, 166.0, 152.0]",5,204.60,225.0,32.0
...,...,...,...,...,...,...,...,...,...
299,11261,Sota Kawasaki,18,3,"[None, None, None, None, None]",0,NaN,15.0,10.0
300,9440,Filippo Mane,3,2,"[None, None, None, None, None]",0,NaN,12.0,4.0
301,10120,Alexander Røssing-Lelesiit,6,4,"[None, None, None, None, None]",0,NaN,10.0,11.0
302,11258,Lazar Jovanovic,9,3,"[None, None, None, None, None]",0,NaN,10.0,3.0


In [3]:
generated_datetime = datetime.now().astimezone()
output_timestamp = generated_datetime.strftime('%Y%m%d_%H%M%S_%z')
output_directory = ensure_directory(DERIVED_KBSTATS_MATCHDAY_QUALIFIED_LAST_5_AVERAGE_PLAYERS_DIR)
json_output_path = output_directory / f'kbstats_matchday_qualified_last_5_average_players_{output_timestamp}.json'
csv_output_path = output_directory / f'kbstats_matchday_qualified_last_5_average_players_{output_timestamp}.csv'
output_document = {
    'generated_at': generated_datetime.isoformat(timespec='seconds'),
    'source_file': selected_input_path.name,
    'eligibility': {'bundesliga_matchday': BUNDESLIGA_MATCHDAY, 'early_season_max_matchday': EARLY_SEASON_MAX_MATCHDAY, 'rule': qualification_rule_description(BUNDESLIGA_MATCHDAY)},
    'last_five_calculation': {'slot_count': MATCH_SLOT_COUNT, 'unplayed_slots_included_in_denominator': False},
    'source_player_count': len(raw_players),
    'valid_player_count': len(valid_players),
    'excluded_invalid_player_count': len(invalid_records),
    'excluded_invalid_last_five_player_count': len(invalid_last_five_records),
    'qualified_player_count': len(qualified_players),
    'players': qualified_players,
}
json_output_path.write_text(json.dumps(output_document, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
csv_df = pd.json_normalize(qualified_players, sep='.')
for column in csv_df.columns:
    csv_df[column] = csv_df[column].map(lambda value: json.dumps(value, ensure_ascii=False) if isinstance(value, (list, dict)) else value)
csv_df.to_csv(csv_output_path, index=False, encoding='utf-8-sig')
print(f'JSON output: {json_output_path}')
print(f'CSV output:  {csv_output_path}')


JSON output: C:\kickbase project\outputs\derived\kbstats_matchday_qualified_last_5_average_players\kbstats_matchday_qualified_last_5_average_players_20260819_085949_+0200.json
CSV output:  C:\kickbase project\outputs\derived\kbstats_matchday_qualified_last_5_average_players\kbstats_matchday_qualified_last_5_average_players_20260819_085949_+0200.csv


In [ ]:
from project_paths import prune_timestamped_outputs

removed_outputs = prune_timestamped_outputs()
print(f"Pruned {len(removed_outputs)} expired timestamped output(s).")
